# Course 5: Winning Online AI Competitions
## Session 5: Tabular competitions: gradient boosting and feature engineering

**Goal:** train LightGBM and XGBoost baselines with out-of-fold predictions, and build smoothed target encoding the right way.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
def make_tabular(n=6000, seed=0):
    """A synthetic binary task with numeric columns, one high-cardinality category, groups and a date."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"id": np.arange(n)})
    for j in range(6): df[f"num{j}"] = rng.normal(size=n)
    n_shops = 300
    shop_effect = rng.normal(scale=1.0, size=n_shops)
    df["shop"] = rng.zipf(1.5, n).clip(max=n_shops) - 1                         # a few big shops, many small ones
    df["customer"] = rng.integers(0, n // 8, n)                                 # about 8 rows per customer
    cust_effect = rng.normal(scale=1.2, size=n // 8)
    df["day"] = np.sort(rng.integers(0, 365, n))
    logit = 0.8 * df.num0 - 0.6 * df.num1 + 0.5 * df.num2 * df.num3 + shop_effect[df["shop"]] + cust_effect[df.customer] + 0.004 * df.day - 1.0
    df["target"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return df
NUM = [f"num{j}" for j in range(6)]

### 1. Baselines with out-of-fold predictions
The same folds for every model. Early stopping picks the number of trees inside each fold.

In [ ]:
import lightgbm as lgb, xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
df = make_tabular(8000); train, test = df.iloc[:6000].reset_index(drop=True), df.iloc[6000:].reset_index(drop=True)
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(train, train.target))

def run_cv(make_fit, feats, tr_df=None, te_df=None):
    tr_df = train if tr_df is None else tr_df; te_df = test if te_df is None else te_df
    oof, pred = np.zeros(len(tr_df)), np.zeros(len(te_df))
    for tr, va in folds:
        predict = make_fit(tr_df.iloc[tr][feats], tr_df.target.iloc[tr], tr_df.iloc[va][feats], tr_df.target.iloc[va])
        oof[va] = predict(tr_df.iloc[va][feats]); pred += predict(te_df[feats]) / len(folds)
    return roc_auc_score(tr_df.target, oof), roc_auc_score(te_df.target, pred), oof, pred

def fit_lgb(Xtr, ytr, Xva, yva):
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=0)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc", callbacks=[lgb.early_stopping(100, verbose=False)])
    return lambda X: m.predict_proba(X)[:, 1]
def fit_xgb(Xtr, ytr, Xva, yva):
    m = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric="auc", early_stopping_rounds=100, random_state=0)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return lambda X: m.predict_proba(X)[:, 1]

results = {}
for name, fn in [("LightGBM", fit_lgb), ("XGBoost", fit_xgb)]:
    results[name] = run_cv(fn, NUM); print(f"{name:9s} numeric features only: CV AUC {results[name][0]:.4f} | test AUC {results[name][1]:.4f}")

### 2. The high-cardinality column
`shop` has 300 values; a few shops have hundreds of rows and many have one or two. Each shop has its own effect on the target. How should a tree model see it?

In [ ]:
counts = train["shop"].value_counts(); print(f"shops: {len(counts)} | rows in the largest: {counts.iloc[0]} | shops with at most 2 rows: {(counts <= 2).sum()}")

### 3. Target encoding, wrong and right
Replace each shop by the mean target of its rows, smoothed towards the global mean so that small shops are not trusted too much:

`enc(c) = (n_c * mean_c + m * global_mean) / (n_c + m)`

where `n_c` is the number of rows of shop c and `m` is the smoothing strength.

- **Wrong:** compute it on all training rows, then cross-validate. Each row's own target is inside its feature.
- **Right:** for the rows of a validation fold, compute it from the other folds only. For the test set, use all training rows.

In [ ]:
def target_encode(fit_df, apply_df, col, m=10):
    g = fit_df.groupby(col).target.agg(["sum", "count"]); prior = fit_df.target.mean()
    enc = (g["sum"] + m * prior) / (g["count"] + m)
    return apply_df[col].map(enc).fillna(prior).values                         # unseen shops get the global mean

naive_tr, naive_te = train.copy(), test.copy()
naive_tr["shop_te"] = target_encode(train, train, "shop", m=1); naive_te["shop_te"] = target_encode(train, test, "shop", m=1)       # WRONG for CV

oof_tr, oof_te = train.copy(), test.copy(); oof_tr["shop_te"] = np.nan
for tr, va in folds: oof_tr.loc[va, "shop_te"] = target_encode(train.iloc[tr], train.iloc[va], "shop", m=10)                        # RIGHT
oof_te["shop_te"] = target_encode(train, test, "shop", m=10)

r_naive = run_cv(fit_lgb, NUM + ["shop_te"], naive_tr, naive_te); r_oof = run_cv(fit_lgb, NUM + ["shop_te"], oof_tr, oof_te)
print(f"no shop feature         : CV {results['LightGBM'][0]:.4f} | test {results['LightGBM'][1]:.4f}")
print(f"naive target encoding   : CV {r_naive[0]:.4f} | test {r_naive[1]:.4f}   <- CV is inflated")
print(f"out-of-fold encoding    : CV {r_oof[0]:.4f} | test {r_oof[1]:.4f}")
assert r_naive[0] > r_naive[1] + 0.005 and r_oof[1] >= r_naive[1] and r_oof[1] > results["LightGBM"][1]

Look at the gaps between CV and test. The naive version promises a score it cannot deliver, and that inflated CV would mislead every later decision (which features to keep, how to weight a blend). It also does worse on the test set, because the model learned to trust a feature that is cleaner in training than it will ever be at prediction time. The out-of-fold version's CV is, if anything, slightly below its test score: each fold's encoding is built from only four fifths of the data. An estimate that errs on the cautious side is the kind you want.

Strictly, the early-stopping fold also uses the encoded value, so careful pipelines nest one more level. The out-of-fold encoding above removes almost all of the leak.

### 4. Two more encodings worth trying

In [ ]:
for d in (oof_tr, oof_te): d["shop_count"] = d["shop"].map(train["shop"].value_counts()).fillna(0).values            # frequency encoding: no target, no leak
r_cnt = run_cv(fit_lgb, NUM + ["shop_te", "shop_count"], oof_tr, oof_te)
def fit_lgb_cat(Xtr, ytr, Xva, yva):
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=31, min_data_per_group=20, cat_smooth=10, verbose=-1, random_state=0)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc", categorical_feature=["shop"], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lambda X: m.predict_proba(X)[:, 1]
r_cat = run_cv(fit_lgb_cat, NUM + ["shop"])
print(f"target + count encoding         : CV {r_cnt[0]:.4f} | test {r_cnt[1]:.4f}")
print(f"LightGBM native categorical     : CV {r_cat[0]:.4f} | test {r_cat[1]:.4f}")

### 5. One hand-made feature
Trees split on one column at a time, so an interaction such as `num2 * num3` takes many splits to approximate. Giving it directly can help more than any tuning.

In [ ]:
for d in (oof_tr, oof_te): d["num2_x_num3"] = d.num2 * d.num3
r_int = run_cv(fit_lgb, NUM + ["shop_te", "num2_x_num3"], oof_tr, oof_te)
print(f"with num2 * num3: CV {r_int[0]:.4f} | test {r_int[1]:.4f}   (before: CV {r_oof[0]:.4f} | test {r_oof[1]:.4f})")

### 6. CatBoost, and GPU data frames (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install catboost`; the cuDF part needs an NVIDIA GPU with RAPIDS (available in Kaggle GPU notebooks).
from catboost import CatBoostClassifier
def fit_cat(Xtr, ytr, Xva, yva):
    m = CatBoostClassifier(iterations=2000, learning_rate=0.03, depth=6, eval_metric="AUC", od_type="Iter", od_wait=100, verbose=0, random_seed=0, cat_features=["shop"])
    m.fit(Xtr, ytr, eval_set=(Xva, yva)); return lambda X: m.predict_proba(X)[:, 1]
# print(run_cv(fit_cat, NUM + ["shop"])[:2])      # CatBoost applies ordered target statistics to categorical columns internally

# cuDF: the pandas API on a GPU, useful when you generate thousands of group-by features.
# %load_ext cudf.pandas        # in a notebook, before importing pandas; then the pandas code above runs on the GPU where supported
# XGBoost on GPU: XGBClassifier(device="cuda", tree_method="hist");  LightGBM: device_type="gpu" (needs a GPU build)

### TODO
1. Sweep the smoothing strength m over 1, 10, 100 with out-of-fold encoding. Plot CV and test AUC.
2. Save the OOF and test predictions of LightGBM and XGBoost in the session 3 format. You will blend them in session 10.
3. Run CatBoost and compare all three libraries on the same folds. Is the difference larger than the fold-to-fold spread?